Connected to Python 3.13.5

In [ ]:
# filepath: c:\Users\kenny\workspace\ContactEstimator\motion_equation.ipynb
import sympy as sp
import numpy as np
from sympy import symbols, Function, Matrix, cos, sin, diff, simplify, sqrt

print("Quadruped Robot Full-Body Dynamics Derivation")
print("=" * 50)

# Step 1: Define all symbolic variables
print("Step 1: Defining symbolic variables...")

# Time variable
t = symbols('t')

# Physical and geometric constants
m_b, m_l = symbols('m_b m_l', positive=True)  # Base mass, leg mass
I_bxx, I_byy = symbols('I_bxx I_byy', positive=True)  # Base inertias
X_offset = symbols('X_offset', positive=True)  # X_offset of hip from base center
Y_offset = symbols('Y_offset', positive=True)  # Y_offset of hip from base center
g = symbols('g', positive=True)  # Gravity acceleration

# Externally time-varying leg rotational inertias.
# The runtime supplies a different I_c value at every sample.
I_c_lf = Function('I_c_lf')(t)
I_c_rf = Function('I_c_rf')(t)
I_c_rh = Function('I_c_rh')(t)
I_c_lh = Function('I_c_lh')(t)

# Generalized coordinates as functions of time
x = Function('x')(t)
z = Function('z')(t)
phi = Function('phi')(t)  # Roll
psi = Function('psi')(t)  # Pitch

# Leg coordinates: beta (rotation angle), Rm (extension length)
beta_lf = Function('beta_lf')(t)
Rm_lf = Function('Rm_lf')(t)
beta_rf = Function('beta_rf')(t)
Rm_rf = Function('Rm_rf')(t)
beta_rh = Function('beta_rh')(t)
Rm_rh = Function('Rm_rh')(t)
beta_lh = Function('beta_lh')(t)
Rm_lh = Function('Rm_lh')(t)

# Generalized coordinates vector (12x1)
q = Matrix([x, z, phi, psi, beta_lf, Rm_lf, beta_rf, Rm_rf, beta_rh, Rm_rh, beta_lh, Rm_lh])

# Generalized velocities and accelerations
q_dot = Matrix([diff(qi, t) for qi in q])
q_ddot = Matrix([diff(qi, t, 2) for qi in q])

print(f"Defined {len(q)} generalized coordinates")

# Define actuator forces/torques as functions of time
print("Defining actuator forces/torques...")
tau_lf_beta = Function('tau_lf_beta')(t)
F_lf_Rm = Function('F_lf_Rm')(t)
tau_rf_beta = Function('tau_rf_beta')(t)
F_rf_Rm = Function('F_rf_Rm')(t)
tau_rh_beta = Function('tau_rh_beta')(t)
F_rh_Rm = Function('F_rh_Rm')(t)
tau_lh_beta = Function('tau_lh_beta')(t)
F_lh_Rm = Function('F_lh_Rm')(t)

# Actuator torque vector (8x1)
tau = Matrix([
    tau_lf_beta, F_lf_Rm,
    tau_rf_beta, F_rf_Rm,
    tau_rh_beta, F_rh_Rm,
    tau_lh_beta, F_lh_Rm
])

print("Actuator torque vector 'tau' created.")
# print(tau.shape)

In [ ]:
q

In [ ]:
# filepath: c:\Users\kenny\workspace\ContactEstimator\motion_equation.ipynb
# Step 2: Kinematic analysis and total kinetic energy calculation
print("Step 2: Computing kinematic analysis and total kinetic energy...")

# Base kinetic energy
K_base = sp.Rational(1,2) * m_b * (diff(x,t)**2 + diff(z,t)**2) + \
         sp.Rational(1,2) * I_bxx * diff(phi,t)**2 + \
         sp.Rational(1,2) * I_byy * diff(psi,t)**2

print("Base kinetic energy computed")

# Base rotation matrix R_b = R_x(phi) * R_y(psi)
R_x_phi = Matrix([
    [1, 0, 0],
    [0, cos(phi), -sin(phi)],
    [0, sin(phi), cos(phi)]
])

R_y_psi = Matrix([
    [cos(psi), 0, sin(psi)],
    [0, 1, 0],
    [-sin(psi), 0, cos(psi)]
])

R_b = R_x_phi * R_y_psi

# sp.pprint(R_b)

# Mounting point vectors (from base center to leg attachment points)
r_H = {
    'lf': Matrix([X_offset, Y_offset, 0]),
    'rf': Matrix([X_offset, -Y_offset, 0]),
    'rh': Matrix([-X_offset, -Y_offset, 0]),
    'lh': Matrix([-X_offset, Y_offset, 0])
}

# Leg parameters dictionary with externally time-varying inertias
legs = {
    'lf': {'beta': beta_lf, 'Rm': Rm_lf, 'I_c': I_c_lf},
    'rf': {'beta': beta_rf, 'Rm': Rm_rf, 'I_c': I_c_rf},
    'rh': {'beta': beta_rh, 'Rm': Rm_rh, 'I_c': I_c_rh},
    'lh': {'beta': beta_lh, 'Rm': Rm_lh, 'I_c': I_c_lh}
}

K_legs = 0
leg_positions = {}

for leg_name, leg_params in legs.items():
    print(f"Processing leg: {leg_name}")
    
    beta = leg_params['beta']
    Rm = leg_params['Rm']
    I_c = leg_params['I_c']
    
    # Leg center of mass relative to mounting point (in base frame)
    p_m_rel = Matrix([-Rm * sin(beta), 0, -Rm * cos(beta)]) 

    # Absolute position of leg center of mass
    base_position = Matrix([x, 0, z])
    p_m_abs = base_position + R_b * (r_H[leg_name] + p_m_rel)
    # print(r_H[leg_name])
    # print(f"  Leg center of mass position: {p_m_abs}")
    
    # Store for potential energy calculation
    leg_positions[leg_name] = p_m_abs
    
    # Absolute velocity of leg center of mass
    v_m_abs = Matrix([diff(p_m_abs[i], t) for i in range(3)])
    
    # Translational kinetic energy of this leg
    K_leg_trans = sp.Rational(1,2) * m_l * (v_m_abs.dot(v_m_abs))
    
    # Rotational kinetic energy with externally time-varying inertia
    # beta and base pitch rotate about the same axis, so the leg absolute angular velocity is dpsi + dbeta.
    K_leg_rot = sp.Rational(1,2) * I_c * (diff(psi, t) + diff(beta, t))**2
    
    # Total kinetic energy for this leg
    K_leg = K_leg_trans + K_leg_rot
    K_legs += K_leg
    
    print(f"  Translational KE:")
    print(sp.latex(sp.simplify(v_m_abs)))
    print(f"  Rotational KE:")
    print(sp.latex(K_leg_rot))

print("Leg kinetic energies (translational + rotational) computed")
print("Note: leg inertias are externally time-varying inputs")

# Total kinetic energy
K_total = K_base + K_legs
print("Total kinetic energy computed")

In [ ]:
K_total_simplified = simplify(K_total)
print("Total kinetic energy simplified:")

In [ ]:
K_total_simplified

In [ ]:
# Step 3: Calculate total potential energy
print("Step 3: Computing total potential energy...")

# Base potential energy
P_base = m_b * g * z

# Legs potential energy
P_legs = 0
for leg_name, p_m_abs in leg_positions.items():
    P_leg = m_l * g * p_m_abs[2]  # Z component
    P_legs += P_leg

# Total potential energy
P_total = P_base + P_legs
print("Total potential energy computed")

In [ ]:
P_total_simplified = simplify(P_total)

In [ ]:
print("Leg positions (absolute coordinates):")
print("=" * 40)

for leg_name, p_m_abs in leg_positions.items():
    print(f"\nLeg {leg_name.upper()}:")
    print(f"  x-coordinate: {p_m_abs[0]}")
    print(f"  y-coordinate: {p_m_abs[1]}")
    print(f"  z-coordinate: {p_m_abs[2]}")

In [ ]:
print("Leg velocity (absolute coordinates):")

for leg_name, p_m_abs in leg_positions.items():
    v_m_abs = Matrix([diff(p_m_abs[i], t) for i in range(3)])
    print(f"\nLeg {leg_name.upper()}:")
    print(f"  x-velocity: {v_m_abs[0]}")
    print(f"  y-velocity: {v_m_abs[1]}")
    print(f"  z-velocity: {v_m_abs[2]}")

In [ ]:
# Step 4: Apply Lagrange-Euler equations
print("Step 4: Applying Lagrange-Euler equations...")

# Lagrangian
L = K_total - P_total

# Calculate equations of motion
EOM_list = []
print("Computing equations of motion for each generalized coordinate...")

for i, qi in enumerate(q):
    print(f"Processing coordinate {i+1}/12: {qi}")
    qi_dot = diff(qi, t)
    
    # Lagrange-Euler equation: d/dt(∂L/∂q̇_i) - ∂L/∂q_i = 0
    dL_dqi_dot = diff(L, qi_dot)
    d_dt_dL_dqi_dot = diff(dL_dqi_dot, t)
    dL_dqi = diff(L, qi)
    
    EOM_i = d_dt_dL_dqi_dot - dL_dqi
    EOM_list.append(EOM_i)

# Create EOM vector
EOM_vec = Matrix(EOM_list)
print("Equations of motion computed")

In [ ]:
# Step 4.5: Define the Actuator Selection Matrix S^T
print("Step 4.5: Defining actuator selection matrix S^T...")

# q is a 12x1 vector. tau is an 8x1 vector. S^T must be a 12x8 matrix.
# It maps the 8 actuator forces to the 12 generalized coordinates.
# The unactuated coordinates (x, z, phi, psi) will have zero rows.

S_T = sp.zeros(12, 8)

# Map beta_lf and Rm_lf torques (tau columns 0, 1) to q rows 4, 5
S_T[4, 0] = 1  # tau_lf_beta -> beta_lf
S_T[5, 1] = 1  # tau_lf_Rm   -> Rm_lf

# Map beta_rf and Rm_rf torques (tau columns 2, 3) to q rows 6, 7
S_T[6, 2] = 1  # tau_rf_beta -> beta_rf
S_T[7, 3] = 1  # tau_rf_Rm   -> Rm_rf

# Map beta_rh and Rm_rh torques (tau columns 4, 5) to q rows 8, 9
S_T[8, 4] = 1  # tau_rh_beta -> beta_rh
S_T[9, 5] = 1  # tau_rh_Rm   -> Rm_rh

# Map beta_lh and Rm_lh torques (tau columns 6, 7) to q rows 10, 11
S_T[10, 6] = 1 # tau_lh_beta -> beta_lh
S_T[11, 7] = 1 # tau_lh_Rm   -> Rm_lh

S = S_T.T

print("Selection matrix S_T created with shape:", S_T.shape)

In [ ]:
# Step 5: Extract M, C, G matrices/vectors using proper Christoffel symbols
print("Step 5: Extracting M, C, G matrices/vectors...")

# Mass matrix M(q): coefficient matrix of q_ddot
print("Computing mass matrix M...")
M = EOM_vec.jacobian(q_ddot)

# Gravity vector G(q): EOM when all velocities and accelerations are zero
print("Computing gravity vector G...")
subs_zero_vel_accel = {}
for qi_dot in q_dot:
    subs_zero_vel_accel[qi_dot] = 0
for qi_ddot in q_ddot:
    subs_zero_vel_accel[qi_ddot] = 0

G = EOM_vec.subs(subs_zero_vel_accel)

# Compute Coriolis matrix C using Christoffel symbols method
print("Computing Coriolis matrix C using Christoffel symbols...")
print("This computation may take some time...")

n = len(q)  # Number of generalized coordinates (12)

# Step 1: Compute time derivative of mass matrix M_dot
print("Computing M_dot (time derivative of mass matrix)...")
M_dot = sp.zeros(n, n)
for i in range(n):
    for j in range(n):
        # Chain rule: dM_ij/dt = sum_k (dM_ij/dq_k * dq_k/dt)
        M_dot_ij = 0
        for k in range(n):
            dM_ij_dqk = diff(M[i, j], q[k])
            M_dot_ij += dM_ij_dqk * q_dot[k]
        M_dot[i, j] = M_dot_ij

print("M_dot computed")

# Step 2: Compute Christoffel symbols c_ijk
print("Computing Christoffel symbols c_ijk...")
# Use dictionary to store 3D tensor c[i][j][k]
christoffel = {}

for i in range(n):
    for j in range(n):
        for k in range(n):
            # c_ijk = 1/2 * (dM_kj/dq_i + dM_ki/dq_j - dM_ij/dq_k)
            dM_kj_dqi = diff(M[k, j], q[i])
            dM_ki_dqj = diff(M[k, i], q[j])
            dM_ij_dqk = diff(M[i, j], q[k])
            
            christoffel[(i, j, k)] = sp.Rational(1, 2) * (dM_kj_dqi + dM_ki_dqj - dM_ij_dqk)

print("Christoffel symbols computed")

# Step 3: Compute Coriolis matrix C
print("Computing Coriolis matrix C...")
C_matrix = sp.zeros(n, n)

for k in range(n):
    for j in range(n):
        # C_kj = sum_i (c_ijk * q_dot_i)
        C_kj = 0
        for i in range(n):
            C_kj += christoffel[(i, j, k)] * q_dot[i]
        C_matrix[k, j] = C_kj

print("Coriolis matrix C computed")

# Compute Coriolis vector C_vec = C * q_dot
C_vec = C_matrix * q_dot

print("Coriolis vector C_vec computed")

# Exact inertia-rate contribution caused by externally varying I_c(t).
D_exact = sp.zeros(n, n)
D_exact[3, 3] = (sp.Derivative(I_c_lf, t) + sp.Derivative(I_c_rf, t) +
                 sp.Derivative(I_c_rh, t) + sp.Derivative(I_c_lh, t))
D_exact[4, 4] = sp.Derivative(I_c_lf, t)
D_exact[6, 6] = sp.Derivative(I_c_rf, t)
D_exact[8, 8] = sp.Derivative(I_c_rh, t)
D_exact[10, 10] = sp.Derivative(I_c_lh, t)

# Runtime approximation: omit the small explicit inertia-rate force.
D_matrix = sp.zeros(n, n)

print("Computed D_exact; exported implementation D is identically zero")

# Exact omitted force, retained for approximation-error verification.
D_exact_vec = D_exact * q_dot
D_vec = D_matrix * q_dot

print("D_vec computed")

In [ ]:
# Check M is symmetric and positive definite
print("Checking mass matrix M properties...")
print("=" * 50)

# 1. Check symmetry: M should equal M^T
print("1. Checking if M is symmetric...")
M_transpose = M.transpose()
M_minus_MT = M - M_transpose
M_symmetry_error = simplify(M_minus_MT)

is_symmetric = M_symmetry_error == sp.zeros(len(q), len(q))
print(f"M is symmetric: {is_symmetric}")

if not is_symmetric:
    print("Non-zero elements in M - M^T:")
    non_zero_count = 0
    for i in range(min(5, len(q))):  # Check first 5x5 submatrix
        for j in range(min(5, len(q))):
            if M_symmetry_error[i, j] != 0:
                print(f"  M[{i},{j}] - M[{j},{i}] = {M_symmetry_error[i, j]}")
                non_zero_count += 1
                if non_zero_count >= 10:  # Limit output
                    break
        if non_zero_count >= 10:
            break

# 2. Check positive definiteness
print("\n2. Checking if M is positive definite...")
print("For a kinetic energy-derived mass matrix, this should always be true")
print("if all masses and inertias are positive.")

# Check diagonal elements (necessary but not sufficient condition)
print("\nDiagonal elements of M:")
diagonal_elements = []
for i in range(len(q)):
    diag_elem = M[i, i]
    diagonal_elements.append(diag_elem)
    print(f"M[{i},{i}] = {diag_elem}")

# For kinetic energy matrices, positive definiteness is guaranteed if:
# - All masses (m_b, m_l) are positive ✓
# - All inertias (I_bxx, I_byy, I_c_*) are positive ✓
# - The kinetic energy expression is properly derived ✓

print("\n3. Physical verification of positive definiteness:")
print("Since M is derived from kinetic energy K = (1/2)q̇ᵀMq̇:")
print("- All physical parameters (masses, inertias) are defined as positive")
print("- The quadratic form represents kinetic energy, which is always ≥ 0")
print("- Therefore, M is positive semidefinite by construction")
print("- M is positive definite if the system has no rigid body modes")

# Check for any zero diagonal elements (which might indicate issues)
zero_diagonals = []
for i, elem in enumerate(diagonal_elements):
    if elem == 0:
        zero_diagonals.append(i)

if zero_diagonals:
    print(f"\nWARNING: Zero diagonal elements found at indices: {zero_diagonals}")
    print("This might indicate rigid body modes or modeling issues")
else:
    print("\n✓ All diagonal elements are non-zero expressions")

print("\nSUMMARY:")
print(f"✓ Symmetric: {is_symmetric}")
print("✓ Positive definite: True (by kinetic energy construction)")
print("✓ Mass matrix M satisfies required properties for robot dynamics")

In [ ]:
# Verify skew-symmetric property of N = M_dot - 2C 
# This property applies to the coordinate-dependent part of M; explicit
# time variation is represented separately by D_exact.
print("\nVerifying skew-symmetric property...")
print("=" * 50)

# Compute N = M_dot - 2*C_matrix
N = M_dot - 2*C_matrix

print("Computing N = M_dot - 2*C...")

# Verify skew-symmetric property: N + N^T should be zero
N_transpose = N.transpose()
N_plus_NT = N + N_transpose

print("Checking if N + N^T = 0 (skew-symmetric property)...")

# Simplify the result
N_plus_NT_simplified = simplify(N_plus_NT)

# Check if it's a zero matrix
is_zero_matrix = N_plus_NT_simplified == sp.zeros(n, n)

print(f"\nSKEW-SYMMETRIC VERIFICATION RESULT:")
print(f"Matrix N dimensions: {N.shape}")
print(f"N + N^T is zero matrix: {is_zero_matrix}")

if is_zero_matrix:
    print("✓ VERIFICATION SUCCESSFUL: N = M_dot - 2C is skew-symmetric")
    print("✓ Coriolis matrix C satisfies the proper mathematical properties")
else:
    print("✗ VERIFICATION FAILED: N = M_dot - 2C is NOT skew-symmetric")
    print("This indicates an error in the Coriolis matrix calculation")
    
    # Show some non-zero elements if verification fails
    non_zero_count = 0
    for i in range(min(3, n)):  # Check first 3x3 submatrix
        for j in range(min(3, n)):
            if N_plus_NT_simplified[i, j] != 0:
                print(f"Non-zero element at [{i},{j}]: {N_plus_NT_simplified[i, j]}")
                non_zero_count += 1
                if non_zero_count >= 5:  # Limit output
                    break
        if non_zero_count >= 5:
            break
    
    if non_zero_count > 0:
        print(f"... and possibly more non-zero elements")

print("\nPhysical interpretation:")
print("The skew-symmetric property N = M_dot - 2C ensures that:")
print("1. Energy conservation in the absence of external forces")
print("2. Proper coupling between kinetic energy and Coriolis effects")
print("3. Passivity of the mechanical system")

In [ ]:
# verify C_vetor is correct
print("\nStep 6: Verifying C_vector is correct")
print("=" * 50)
print()
C_vec_direct = EOM_vec - M * q_ddot - G
# Check if C_vec matches the direct computation
C_vec_error = (C_vec + D_exact_vec) - C_vec_direct

C_vec_error_simplified = simplify(C_vec_error)
print("C_vector is correct" if C_vec_error_simplified == sp.zeros(n, 1) else "C_vector is NOT correct")

C_matrix_simplified = simplify(C_matrix)


In [ ]:
C_vec_simplified = simplify(C_vec)

In [ ]:
C_vec_simplified

In [ ]:
C_matrix_simplified

In [ ]:
C_vec_error_simplified = simplify(C_vec_error)

In [ ]:
C_vec_error_simplified

In [ ]:
D_vec

In [ ]:
G_simplified = simplify(G)

In [ ]:
G_simplified

In [ ]:
M_simplified = simplify(M)

In [ ]:
M_simplified

In [ ]:
D_matrix

In [ ]:
# Store all equations in a dictionary
equations_dict = {
    'M': M_simplified,
    'C': C_matrix_simplified,
    'D': D_matrix,
    'D_exact': D_exact,
    'G': G_simplified,
    'S_T': S_T,
    'q': q,
    'q_dot': q_dot,
    'q_ddot': q_ddot,
    'tau': tau,
    'symbols': {
        'm_b': m_b, 'm_l': m_l, 'I_bxx': I_bxx, 'I_byy': I_byy, 'X_offset': X_offset, 'Y_offset': Y_offset, 'g': g, 't': t
    },
    'functions': {
        'x': x, 'z': z, 'phi': phi, 'psi': psi, 
        'beta_lf': beta_lf, 'Rm_lf': Rm_lf, 'I_c_lf': I_c_lf,
        'beta_rf': beta_rf, 'Rm_rf': Rm_rf, 'I_c_rf': I_c_rf,
        'beta_rh': beta_rh, 'Rm_rh': Rm_rh, 'I_c_rh': I_c_rh,
        'beta_lh': beta_lh, 'Rm_lh': Rm_lh, 'I_c_lh': I_c_lh,
        'tau_lf_beta': tau_lf_beta, 'F_lf_Rm': F_lf_Rm,
        'tau_rf_beta': tau_rf_beta, 'F_rf_Rm': F_rf_Rm,
        'tau_rh_beta': tau_rh_beta, 'F_rh_Rm': F_rh_Rm,
        'tau_lh_beta': tau_lh_beta, 'F_lh_Rm': F_lh_Rm
    }
}

In [ ]:
import time
from pathlib import Path
import pickle

# Create directory for saved equations
save_dir = Path('../saved_equations')
save_dir.mkdir(exist_ok=True)
print("Saving with dill...")
start_time = time.time()

try:
    portable_equations = {'serialization': 'sympy-srepr-v1'}
    for key, value in equations_dict.items():
        if key in {'symbols', 'functions'}:
            portable_equations[key] = {
                name: sp.srepr(expression) for name, expression in value.items()
            }
        elif key == 'model_assumptions':
            portable_equations[key] = value
        else:
            portable_equations[key] = sp.srepr(value)

    with open(save_dir / 'equations_dill.pkl', 'wb') as f:
        pickle.dump(portable_equations, f, protocol=4)
    
    pickle_save_time = time.time() - start_time
    print(f"✓ Pickle save successful in {pickle_save_time:.4f} seconds")
    print(f"File size: {(save_dir / 'equations_dill.pkl').stat().st_size / 1024:.2f} KB")
except Exception as e:
    print(f"✗ Dill save failed: {e}")

In [ ]:
S_T